# Brain2Text25 long POSSM reconstruction and transfer

Colab-first workflow for the longer reconstruction experiment. The primary SSL run uses brain2text25 train/val neural recordings only, with labels ignored. Stage 2 evaluates the same POSSM-GRU CTC decoder from reconstruction initialization and from random initialization on the official source `train -> val` split.

Use the raw cache for Stage 2 and the sigma-2 pre-smoothed cache for Stage 1. Keep the test split untouched.

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive')
RAW_CACHE_ROOT = DRIVE_ROOT / 'utah_ssl' / 'data' / 'cache_v1'
STAGE1_CACHE_ROOT = DRIVE_ROOT / 'utah_ssl' / 'data' / 'cache_v1_smoothed_sigma2p0'
OUTPUT_ROOT = DRIVE_ROOT / 'utah_ssl' / 'outputs' / 'ssl_experiments' / 'brain2text25_long_pretraining'
STATS_ROOT = DRIVE_ROOT / 'utah_ssl' / 'data' / 'stats'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('raw:', RAW_CACHE_ROOT, RAW_CACHE_ROOT.exists())
print('stage1:', STAGE1_CACHE_ROOT, STAGE1_CACHE_ROOT.exists())
print('output:', OUTPUT_ROOT)

In [ ]:
import os
import subprocess
import sys

REPO_URL = 'https://github.com/ethan-read/utah-ssl.git'
REPO_DIR = Path('/content/utah-ssl')
EXPERIMENTS_DIR = REPO_DIR / 'analysis' / 'active' / 'ssl_experiments'
POSSM_DIR = REPO_DIR / 'analysis' / 'reference' / 'possm'
BENCHMARK_DIR = REPO_DIR / 'analysis' / 'active' / 'transfer_benchmark' / 'ssl_autoresearch'
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
for import_path in (REPO_DIR, EXPERIMENTS_DIR, POSSM_DIR, BENCHMARK_DIR):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))
os.environ['SSL_AUTORESEARCH_CACHE_ROOT'] = str(RAW_CACHE_ROOT)
print('repo:', REPO_DIR)

In [ ]:
import importlib
import json
import subprocess
import time

import torch

from possm_ssl import (
    CacheAccessConfig, POSSMFinetuneConfig, POSSMTrainingConfig,
    display_possm_stage1_report, display_possm_stage2_report,
    prepare_cache_context, recover_possm_run_state_from_checkpoint,
    resolve_latest_possm_checkpoint_path, resume_possm_training,
    run_possm_phoneme_finetuning, run_possm_training,
)
from masked_ssl.cache import load_cache_smoothing_provenance

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 7
DATASET = 'brain2text25'
FEATURE_MODE = 'tx_sbp'
BOUNDARY_KEY_MODE = 'session'
SEGMENT_BINS = 100
PRETRAIN_SOURCE_SPLITS = ('train', 'val')
STAGE1_STEPS = 60000
STAGE1_RUN_NAME = 'brain2text25_possm_stage1_tx_sbp_60k_seed7'
STAGE2_STEPS = 12000
print('device:', DEVICE)

## Cache and split validation

This cell checks that the cache is the area-6v brain2text25 contract and that the held-out test split is not used by Stage 1.

In [ ]:
def validate_b2t25_root(root):
    dataset_root = Path(root) / DATASET
    metadata = json.loads((dataset_root / 'metadata.json').read_text())
    rows = [json.loads(line) for line in (dataset_root / 'manifest.jsonl').read_text().splitlines() if line.strip()]
    assert dataset_root.is_dir(), dataset_root
    assert (dataset_root / 'shards').is_dir()
    assert all(int(row['n_tx_features']) == 128 for row in rows)
    assert all(int(row['n_sbp_features']) == 128 for row in rows)
    counts = {}
    for row in rows:
        counts[row['source_split']] = counts.get(row['source_split'], 0) + 1
    return {'root': str(root), 'examples': len(rows), 'split_counts': counts,
            'sessions': len({row['session_id'] for row in rows}),
            'time_bins': sum(int(row['n_time_bins']) for row in rows),
            'metadata_smoothing': metadata.get('smoothing_provenance')}

raw_inventory = validate_b2t25_root(RAW_CACHE_ROOT)
stage1_inventory = validate_b2t25_root(STAGE1_CACHE_ROOT)
assert raw_inventory['split_counts'].get('test', 0) > 0
assert set(PRETRAIN_SOURCE_SPLITS).isdisjoint({'test'})
print(json.dumps({'raw': raw_inventory, 'stage1': stage1_inventory}, indent=2))
smoothing = load_cache_smoothing_provenance(STAGE1_CACHE_ROOT, dataset=DATASET)
if smoothing is None:
    raise ValueError('Stage-1 cache has no smoothing provenance; use the Drive sigma-2 cache.')
print('stage1 smoothing provenance:', smoothing)

## Prepare exact normalization artifacts

Session stats are computed from brain2text25 train/val only. Stage-2 stats are computed from source `train` and reused for validation.

In [ ]:
FORCE_RECOMPUTE_STATS = False
SESSION_STATS_PATH = STATS_ROOT / 'session_feature_stats' / 'smoothed_sigma2p0' / 'tx_sbp' / 'session' / 'ssl_pretrain_brain2text25_train_val_v1.pt'
SPLIT_STATS_PATH = STATS_ROOT / 'split_feature_stats' / 'raw' / DATASET / 'train' / FEATURE_MODE / 'global_v1.pt'

def ensure_artifact(label, path, command):
    path = Path(path)
    if path.exists() and path.with_suffix('.json').exists() and not FORCE_RECOMPUTE_STATS:
        print('reusing', label, path)
        return
    command = [str(part) for part in command]
    if FORCE_RECOMPUTE_STATS or path.exists() or path.with_suffix('.json').exists():
        command.append('--overwrite')
    print('running', label, ':', ' '.join(command))
    subprocess.run(command, cwd=REPO_DIR, check=True)

ensure_artifact('session stats', SESSION_STATS_PATH, [
    sys.executable, 'analysis/active/ssl_experiments/ssl_core/scripts/recompute_session_feature_stats.py',
    '--cache-root', STAGE1_CACHE_ROOT, '--output-path', SESSION_STATS_PATH,
    '--dataset', DATASET, '--feature-mode', FEATURE_MODE, '--boundary-key-mode', BOUNDARY_KEY_MODE,
    '--tx-dim', 128, '--sbp-dim', 128, '--segment-bins', SEGMENT_BINS,
    '--source-split', 'train', '--source-split', 'val',
])
ensure_artifact('source train-val split stats', SPLIT_STATS_PATH, [
    sys.executable, 'analysis/active/ssl_experiments/ssl_core/scripts/recompute_split_feature_stats.py',
    '--cache-root', RAW_CACHE_ROOT, '--output-path', SPLIT_STATS_PATH,
    '--dataset', DATASET, '--feature-mode', FEATURE_MODE, '--boundary-key-mode', BOUNDARY_KEY_MODE,
    '--split-policy', 'source_train_val',
])
print('session stats:', SESSION_STATS_PATH)
print('split stats:', SPLIT_STATS_PATH)

## Build the train/val-only Stage-1 context

In [ ]:
available_stage1_datasets = tuple(path.name for path in STAGE1_CACHE_ROOT.iterdir() if path.is_dir() and (path / 'metadata.json').exists())
excluded_stage1_datasets = tuple(name for name in available_stage1_datasets if name != DATASET)
CACHE_ACCESS_CONFIG = CacheAccessConfig(
    mode='drive_direct', local_cache_base='/content/utah_ssl_cache',
    excluded_datasets=excluded_stage1_datasets, seed=SEED, segment_bins=SEGMENT_BINS,
    use_normalization=True, examples_per_shard=8, tx_dim=128, sbp_dim=128,
    feature_mode=FEATURE_MODE, boundary_key_mode=BOUNDARY_KEY_MODE,
    gaussian_smoothing_sigma_bins=0.0,
    precomputed_session_stats_path=SESSION_STATS_PATH,
    pretrain_source_splits=PRETRAIN_SOURCE_SPLITS,
)
CACHE_CONTEXT = prepare_cache_context(cache_candidates=[STAGE1_CACHE_ROOT], config=CACHE_ACCESS_CONFIG)
print('pretrain datasets:', CACHE_CONTEXT.pretrain_datasets)
print('source split filter:', CACHE_CONTEXT.config.pretrain_source_splits)
print('session split summary:', CACHE_CONTEXT.session_split_summary[DATASET])

## Long Stage-1 reconstruction

The cell is resumable. Set `STAGE1_ACTION='train'` for a fresh run, or `STAGE1_ACTION='resume'` to extend the existing run to `STAGE1_STEPS`.

In [ ]:
STAGE1_ACTION = 'resume'  # 'train' or 'resume'
STAGE1_CONFIG = POSSMTrainingConfig(
    seed=SEED, data_mode='normalized', feature_mode=FEATURE_MODE, boundary_key_mode=BOUNDARY_KEY_MODE,
    segment_bins=SEGMENT_BINS, model_dim=64, latent_count=4, value_encoder_type='linear',
    ffn_hidden_size=512, dropout=0.15, use_token_norm=True, batch_size=32, num_steps=STAGE1_STEPS,
    learning_rate=3e-4, weight_decay=1e-3, val_every=500, val_batches=10,
    checkpoint_every_steps=2000, checkpoint_keep_last=3, dataset_weight_alpha=0.25,
    examples_per_shard=8, log_every=50, temporal_backbone_type='gru',
    temporal_gru_num_layers=1, temporal_gru_dropout=0.0, temporal_gru_bidirectional=False,
    stage1_objective_type='plain_mse', masking_type='none', mask_prob=0.0,
    reconstruction_head_type='linear', reconstruction_mlp_hidden_size=None,
)
STAGE1_OUTPUT_ROOT = OUTPUT_ROOT / 'stage1'
STAGE1_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
if STAGE1_ACTION == 'train':
    STAGE1_RUN_STATE = run_possm_training(cache_context=CACHE_CONTEXT, config=STAGE1_CONFIG,
        output_root=STAGE1_OUTPUT_ROOT, device=DEVICE, run_name=STAGE1_RUN_NAME)
elif STAGE1_ACTION == 'resume':
    checkpoint = resolve_latest_possm_checkpoint_path(output_root=STAGE1_OUTPUT_ROOT, run_dir=STAGE1_OUTPUT_ROOT / STAGE1_RUN_NAME)
    STAGE1_RUN_STATE = recover_possm_run_state_from_checkpoint(cache_context=CACHE_CONTEXT, checkpoint_path=checkpoint, device=DEVICE)
    current_step = int(STAGE1_RUN_STATE['checkpoint_step'])
    if current_step >= STAGE1_STEPS:
        print('Stage 1 already reached', current_step, 'steps')
    else:
        STAGE1_RUN_STATE = resume_possm_training(run_state=STAGE1_RUN_STATE,
            additional_steps=STAGE1_STEPS - current_step, cache_context=CACHE_CONTEXT, device=DEVICE)
else:
    raise ValueError(STAGE1_ACTION)
print('stage1 run:', STAGE1_RUN_STATE['run_dir'])
print('best checkpoint:', STAGE1_RUN_STATE['best_checkpoint_path'])
display_possm_stage1_report(STAGE1_RUN_STATE)

## Stage-2 pretrained-versus-random transfer

This uses the source `train -> val` split, raw cache, train-derived global stats, online normalize/augment/smooth ordering, and the POSSM-GRU post-decoder emission head that produced the strongest prior transfer result.

In [ ]:
STAGE1_CHECKPOINT = Path(STAGE1_RUN_STATE['best_checkpoint_path'])
STAGE2_OUTPUT_ROOT = OUTPUT_ROOT / 'stage2'
STAGE2_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def run_stage2(init_source):
    config = POSSMFinetuneConfig(
        seed=SEED, mode='finetune_full', init_source=init_source, dataset=DATASET,
        split_policy='source_train_val', feature_mode=FEATURE_MODE, data_mode='normalized',
        boundary_key_mode=BOUNDARY_KEY_MODE, batch_size=32, num_steps=STAGE2_STEPS,
        learning_rate=2e-4, encoder_learning_rate=3e-5, weight_decay=1e-3,
        max_grad_norm=1.0, val_every_steps=100, checkpoint_every_steps=500,
        progress_every_steps=25, session_adapter_enabled=False,
        input_smoothing_sigma_bins=2.0, input_smoothing_kernel_size=100,
        input_smoothing_threshold=0.01, white_noise_sd=0.1, constant_offset_sd=0.05,
        gru_hidden_size=768, gru_num_layers=5, gru_dropout=0.2,
        emission_mode='post_decoder_conv', conv_kernel_size=14, conv_stride=4, conv_dropout=0.1,
        precomputed_split_stats_path=SPLIT_STATS_PATH,
    )
    return run_possm_phoneme_finetuning(
        checkpoint_path=STAGE1_CHECKPOINT, cache_root=RAW_CACHE_ROOT,
        output_root=STAGE2_OUTPUT_ROOT, config=config, device=DEVICE,
        run_name=f'brain2text25_possm_gru_{init_source}_seed{SEED}',
        resume_from_latest=True,
    )

RUN_PRETRAINED_STAGE2 = True
RUN_RANDOM_STAGE2 = True
if RUN_PRETRAINED_STAGE2:
    STAGE2_PRETRAINED = run_stage2('stage1')
    display_possm_stage2_report(STAGE2_PRETRAINED)
if RUN_RANDOM_STAGE2:
    STAGE2_RANDOM = run_stage2('random')
    display_possm_stage2_report(STAGE2_RANDOM)

## Interpretation

Use validation PER, CTC, blank-frame rate, prediction/reference length ratio, and per-session results. Treat reconstruction loss as a diagnostic only. The primary success criterion is consistent downstream improvement from `stage1` initialization over the matched `random` run.